# Customers Ireland Materialized View

Create a materialized view of deduplicated customers located in Ireland, using the golden source table.

**Lineage:** `CUSTOMERS_GOLDEN` → `CUSTOMERS_IRELAND`

In [ ]:
import uuid
from datetime import datetime

run_id = str(uuid.uuid4())[:8]
notebook_name = '03_customers_ireland_matview'
start_time = datetime.now()
print(f"Pipeline Run ID: {run_id}")

In [ ]:
%%sql -r create_ireland_matview
CREATE OR REPLACE MATERIALIZED VIEW LANSDOWNEPARTNERS_DB.CORE.CUSTOMERS_IRELAND AS
SELECT
    GOLDEN_ID,
    CUSTOMER_ID,
    FULL_NAME,
    COMPANY_NAME,
    INVESTOR_TYPE,
    REGION,
    COUNTRY,
    AUM_COMMITMENT_GBP,
    RELATIONSHIP_START_DATE,
    RELATIONSHIP_MANAGER,
    RISK_PROFILE,
    EMAIL,
    STATUS
FROM LANSDOWNEPARTNERS_DB.CORE.CUSTOMERS_GOLDEN
WHERE IS_MASTER_RECORD = TRUE
  AND UPPER(COUNTRY) IN ('IRELAND', 'REPUBLIC OF IRELAND', 'EIRE')

In [ ]:
%%sql -r verify_ireland
SELECT COUNT(*) AS ireland_customer_count
FROM LANSDOWNEPARTNERS_DB.CORE.CUSTOMERS_IRELAND

## Record Lineage

In [ ]:
from snowflake.snowpark.context import get_active_session
from datetime import datetime

session = get_active_session()
end_time = datetime.now()
duration = int((end_time - start_time).total_seconds())

row_count = session.sql("SELECT COUNT(*) AS cnt FROM LANSDOWNEPARTNERS_DB.CORE.CUSTOMERS_IRELAND").collect()[0]['CNT']

session.sql(f"""
    INSERT INTO LANSDOWNEPARTNERS_DB.CORE.PIPELINE_LINEAGE 
    (RUN_ID, NOTEBOOK_NAME, STEP_NAME, SOURCE_OBJECT, TARGET_OBJECT, OPERATION, ROW_COUNT, STATUS, DURATION_SECONDS)
    VALUES (
        '{run_id}',
        '{notebook_name}',
        'create_ireland_matview',
        'LANSDOWNEPARTNERS_DB.CORE.CUSTOMERS_GOLDEN',
        'LANSDOWNEPARTNERS_DB.CORE.CUSTOMERS_IRELAND',
        'CREATE MATERIALIZED VIEW',
        {row_count},
        'SUCCESS',
        {duration}
    )
""").collect()

print(f"Lineage recorded: CUSTOMERS_GOLDEN -> CUSTOMERS_IRELAND ({row_count} rows, {duration}s)")